# Analisis Exploratorio Continuacion
## Mario Alejandro Castro Lerma

Continuacion de EDA a partir del dataset rick_eda_imput.csv generado por Esthefania en 2.1-eot-EDA_PCA.ipynb

---

### Analisis rapido

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/interim/rick_eda_imput.csv")
# df = pd.read_csv("../data/interim/rick_hist.csv")

In [3]:
df.head()

,ide_id,ide_sex,ide_eda_ano,des_cal,ide_col,des_jur_res,des_mpo_res,cve_loc_res,des_loc_res,ide_cp,...,dias_estudio,demora_pac,demora_pac_fiebre,demora_pac_signos_alarma,fec_ini_estudio_corr,fec_fin_estudio_corr,fec_sol_aten_corr,fec_ini_signos_alarma_corr,fec_ini_signos_sint_corr,fec_ini_fiebre_corr
0,928424,1,39,CALLE,PASEOS DEL PEDREGAL Fraccionamiento,HERMOSILLO,HERMOSILLO,289,HERMOSILLO,83118.0,...,7.0,2,2.0,1.0,False,False,False,False,False,False
1,928745,2,8,CALLE,SAN RAFAEL Colonia,SAN LUIS RIO COLORADO,PUERTO PEÑASCO,1,PUERTO PEÑASCO,83557.0,...,NaN,1,1.0,NaN,False,False,False,False,False,False
2,928768,2,2,CALLE,EL BOSQUE (CALLE QUINCE) Ejido,CAJEME,CAJEME,365,QUETCHEHUECA,85207.0,...,0.0,3,3.0,NaN,False,False,False,False,False,False
3,929029,1,9,PRIVADA,PUERTO PEÑASCO CENTRO Colonia,SAN LUIS RIO COLORADO,PUERTO PEÑASCO,1,PUERTO PEÑASCO,83550.0,...,NaN,3,2.0,NaN,False,False,False,False,False,False
4,929184,2,13,CALLE,PALO BLANCO Congregación,NAVOJOA,NAVOJOA,124,TESIA,85235.0,...,24.0,8,8.0,NaN,False,False,False,False,False,False


Por nota de Esthefania eliminare _corr, _imput, _uni_trat y otras columnas

In [4]:
cols_to_remove = [col for col in df.columns if col.endswith(('_corr', '_imput', '_uni_trat'))]
cols_to_remove += ['ide_cp', 'des_diag_final', 'compromiso_organos'] 
print(f"Columns to remove: {cols_to_remove}")
df = df.drop(columns=cols_to_remove)

Columns to remove: ['des_jur_uni_trat', 'des_mpo_uni_trat', 'des_ins_uni_trat', 'des_cual_lengua_imput', 'otros_especifique_imput', 'contacto_otr_imput', 'otro_sintoma_imput', 'otr_serv_atencion_imput', 'temperatura_imput', 'fec_ini_estudio_corr', 'fec_fin_estudio_corr', 'fec_sol_aten_corr', 'fec_ini_signos_alarma_corr', 'fec_ini_signos_sint_corr', 'fec_ini_fiebre_corr', 'ide_cp', 'des_diag_final', 'compromiso_organos']


In [5]:
df.shape

(2631, 172)

Columnas con vacios:

In [6]:
for col in df.columns:
    if df[col].isna().any():
        print(f"{col}: {df[col].isna().sum()}")
        

fec_ini_estudio: 2
fec_fin_estudio: 1265
fec_ini_fiebre: 80
fec_ini_signos_alarma: 2211
fec_ingreso: 1153
cve_uni_med_tratante: 810
des_unidad_tratante: 810
cve_diag_hospit: 1325
fec_egreso: 1549
fec_defuncion: 2291
fecha_toma_rickett_ser1: 1382
fecha_recep_rickett_ser1: 1610
fecha_resultado_rickett_ser1: 1640
rickett_resultado_ser1: 191
fecha_toma_rickett_sangre: 300
fecha_recep_rickett_sangre: 381
mstra_rech_ricket_sangre: 191
motiv_rech_ricket_sangre: 191
ricket_cq_lesp: 191
ricket_cq: 191
res_final_ricket_inmuno: 191
res_final_ricket_rtpcr: 191
fecha_resultado_rickett_rtpcr: 406
ricket_especie_rtpcr: 191
fec_ini_trat_ricket: 655
fec_fin_trat_ricket: 1806
des_ins_uni_trat_norm: 704
tiempo_res_hosp: 1
dias_estudio: 1263
demora_pac_fiebre: 80
demora_pac_signos_alarma: 2211


Dropeare las columnas con mas de 80% de faltantes

In [7]:
# Percentage threshold
threshold = 0.80
cols_to_drop = df.columns[df.isnull().mean() > threshold]

# Drop them
df = df.drop(columns=cols_to_drop)

# Optional: see which columns were removed
print("Dropped columns:")
print(list(cols_to_drop))

Dropped columns:
['fec_ini_signos_alarma', 'fec_defuncion', 'demora_pac_signos_alarma']


In [8]:
df.shape

(2631, 169)

---

### Separando el dataset en 2, uno para la prediccion de rickettsia y otro para predecir la letalidad

In [13]:
easy_access_cols = [

    # Demographics
    "ide_sex",
    "ide_eda_ano",
    "es_indigena",

    # Exposure
    "contacto_garrapata",
    "contacto_otr",
    "agua_potable",
    "eliminacion_basura",

    # Symptoms / signs
    "temperatura",
    "cefalea",
    "mialgias",
    "artralgias",
    "exantema",
    "nauseas",
    "vomito",
    "petequias",
    "dolor_abdominal_intenso",
    "letargo",
    "irritabilidad",
    "taquicardia",
    "fotofobia",
    "diarrea",
    "conjuntivitis",
    "tos",
    "faringitis",
    "ictericia",
    "convulsiones",
    "somolencia",
    "adinamia",
    "astenia",
    "cefalea_frontal",
    "perdida_peso",
    "escalofrios",
    "diaforesis",
    "anorexia",

    # Comorbidities
    "diabetes",
    "hipertension",
    "enf_ulcero_peptica",
    "enf_renal",
    "inmunosupresion",
    "cirrosis_hepatica",
    "embarazo",
    "sem_gest",

    # Temporal progression
    "demora_pac",
    "demora_pac_fiebre",
]

target_col = "estatus_caso"

prediction_df = df[easy_access_cols + [target_col]]

In [14]:
prediction_df.head() 

,ide_sex,ide_eda_ano,es_indigena,contacto_garrapata,contacto_otr,agua_potable,eliminacion_basura,temperatura,cefalea,mialgias,...,hipertension,enf_ulcero_peptica,enf_renal,inmunosupresion,cirrosis_hepatica,embarazo,sem_gest,demora_pac,demora_pac_fiebre,estatus_caso
0,1,39,2,1,Se ignora,2,2,39.0,2,1,...,0,0,0,0,0,0,0,2,2.0,2
1,2,8,2,1,Se ignora,1,2,39.0,1,1,...,0,0,0,0,0,0,0,1,1.0,3
2,2,2,2,0,Se ignora,1,1,39.0,1,1,...,0,0,0,0,0,0,0,3,3.0,1
3,1,9,2,1,Se ignora,1,1,39.0,1,1,...,0,0,0,0,0,0,0,3,2.0,3
4,2,13,2,1,Se ignora,1,1,39.0,1,1,...,0,0,0,0,0,0,0,8,8.0,2


In [15]:
for col in prediction_df.columns:
    if prediction_df[col].isna().any():
        print(f"{col}: {prediction_df[col].isna().sum()}")

demora_pac_fiebre: 80


In [16]:
prediction_df['demora_pac_fiebre'].describe()

count    2551.000000
mean        2.914543
std         2.735631
min        -3.000000
25%         1.000000
50%         3.000000
75%         4.000000
max        63.000000
Name: demora_pac_fiebre, dtype: float64

Como no sabemos la demora en esos 80 pacientes, he decidido utilizar la media.

In [17]:
prediction_df.fillna(value={'demora_pac_fiebre': prediction_df['demora_pac_fiebre'].mean()}, inplace=True)

,ide_sex,ide_eda_ano,es_indigena,contacto_garrapata,contacto_otr,agua_potable,eliminacion_basura,temperatura,cefalea,mialgias,...,hipertension,enf_ulcero_peptica,enf_renal,inmunosupresion,cirrosis_hepatica,embarazo,sem_gest,demora_pac,demora_pac_fiebre,estatus_caso
0,1,39,2,1,Se ignora,2,2,39.0,2,1,...,0,0,0,0,0,0,0,2,2.0,2
1,2,8,2,1,Se ignora,1,2,39.0,1,1,...,0,0,0,0,0,0,0,1,1.0,3
2,2,2,2,0,Se ignora,1,1,39.0,1,1,...,0,0,0,0,0,0,0,3,3.0,1
3,1,9,2,1,Se ignora,1,1,39.0,1,1,...,0,0,0,0,0,0,0,3,2.0,3
4,2,13,2,1,Se ignora,1,1,39.0,1,1,...,0,0,0,0,0,0,0,8,8.0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2626,2,20,2,1,Se ignora,1,2,39.0,1,1,...,0,0,0,0,0,0,0,2,2.0,2
2627,2,24,2,1,Se ignora,1,1,39.0,1,2,...,0,0,0,0,0,0,0,3,3.0,3
2628,2,20,2,1,Se ignora,1,1,39.0,1,1,...,0,0,0,0,0,0,0,8,8.0,2
2629,1,1,2,1,Se ignora,0,0,38.0,1,2,...,0,0,0,0,0,0,0,2,2.0,3


In [18]:
for col in prediction_df.columns:
    if prediction_df[col].isna().any():
        print(f"{col}: {prediction_df[col].isna().sum()}")

Revisando la columna con str:

In [27]:
prediction_df['contacto_otr'].value_counts()

contacto_otr
se ignora                    2317
perros                        229
perros con ectoparásitos       20
perros y gatos                 19
sin contacto                   14
perros con garrapatas           9
roedores                        4
pulgas                          3
gatos                           3
garrapatas                      3
zoonosis                        2
insectos no especificados       2
exposición ambiental            2
perros y caballos               1
conejos                         1
otros animales                  1
arañas                          1
Name: count, dtype: int64

In [26]:
prediction_df['contacto_otr'] = prediction_df['contacto_otr'].str.lower()

### Aplicando modelos